# Darkroom Studio

A deterministic, non-generative photo colour-grading and editing studio for Google Colab.

* Every edit is a **validated JSON recipe** executed by NumPy/OpenCV code in this repository. Nothing redraws, invents or removes image content.
* Natural-language prompts are turned into recipes by **local rules** (offline, default) or, optionally, by an **OpenRouter LLM that only ever receives text**. The photograph is never sent anywhere, and LLM output is validated against a schema and never executed.
* Originals (including RAW files) are copied to a read-only library and never modified. Exports are rendered from the float32 full-resolution source, never from the preview.

Run the cells in order. Cells 1–4 set up the runtime, cells 5–11 load and self-check each subsystem, cell 12 runs the automated tests, and cell 13 opens the app.

**This notebook is a new working file.** It does not overwrite any earlier notebook; cell 4 backs up an existing checkout before updating it.

## 1 · Environment inspection

In [ ]:
import os, sys, platform, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print("Python  ", sys.version.split()[0], "|", platform.platform())
print("Colab   ", IN_COLAB)
print("CPUs    ", os.cpu_count())
try:
    with open("/proc/meminfo") as f:
        mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
    print(f"RAM      {mem['MemTotal']/2**20:.1f} GB total, {mem['MemAvailable']/2**20:.1f} GB available")
except Exception:
    pass
du = shutil.disk_usage("/content" if IN_COLAB else ".")
print(f"Disk     {du.free/2**30:.0f} GB free")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
print("GPU     ", gpu.stdout.strip() if gpu and gpu.returncode == 0 else "none (the editor runs on CPU; a GPU is not required)")

## 2 · Dependencies (guarded install)

Only installs what is missing or outside the tested range, so re-running is cheap. It never upgrades NumPy, SciPy or PyTorch that Colab ships, and never installs diffusion models. ExifTool is installed with `apt` for full metadata copying; without it the app falls back to a basic EXIF writer.

In [ ]:
import importlib.metadata as md, subprocess, sys, shutil

def have(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None

def vtuple(v):
    out = []
    for p in v.split(".")[:3]:
        n = "".join(ch for ch in p if ch.isdigit())
        out.append(int(n) if n else 0)
    return tuple(out)

# (distribution, minimum tested, pip spec used only when missing or too old)
NEEDS = [
    ("numpy",                  "2.0.2",     None),                 # Colab's own; never touched
    ("scipy",                  "1.14.1",    None),                 # Colab's own; never touched
    ("pillow",                 "11.0.0",    None),                 # Colab's own; never touched
    ("tifffile",               "2024.8.30", "tifffile>=2024.8.30"),
    ("rawpy",                  "0.24.0",    "rawpy>=0.24,<0.29"),
    ("exifread",               "3.0.0",     "exifread>=3.0,<4"),
    ("gradio",                 "6.2.0",     "gradio>=6.2,<7"),
]
cv = have("opencv-python-headless") or have("opencv-python") or have("opencv-contrib-python")
todo = [spec for dist, lo, spec in NEEDS if spec and (have(dist) is None or vtuple(have(dist)) < vtuple(lo))]
if cv is None:
    todo.append("opencv-python-headless>=4.10")
for dist, lo, spec in NEEDS:
    if spec is None and (have(dist) is None or vtuple(have(dist)) < vtuple(lo)):
        print(f"WARNING: {dist} {have(dist)} is older than the tested {lo}; not upgrading it automatically.")
if todo:
    print("pip install", *todo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *todo], check=True)
else:
    print("Python packages: already satisfied, nothing installed.")

if shutil.which("exiftool") is None:
    if shutil.which("apt-get"):
        print("Installing ExifTool (apt) ...")
        subprocess.run("apt-get -qq update && apt-get -qq install -y libimage-exiftool-perl > /dev/null",
                       shell=True, check=False)
    else:
        print("ExifTool not available: metadata copying falls back to a basic EXIF block.")
print("exiftool:", shutil.which("exiftool") or "not installed")

## 3 · Version verification

In [ ]:
import importlib, shutil, subprocess
rows = []
for mod, dist in [("numpy", "numpy"), ("scipy", "scipy"), ("PIL", "pillow"), ("cv2", None), ("tifffile", "tifffile"),
                  ("rawpy", "rawpy"), ("exifread", "exifread"), ("gradio", "gradio")]:
    m = importlib.import_module(mod)
    rows.append((mod, getattr(m, "__version__", "?")))
import rawpy
rows.append(("LibRaw", ".".join(map(str, rawpy.libraw_version))))
if shutil.which("exiftool"):
    rows.append(("ExifTool", subprocess.run(["exiftool", "-ver"], capture_output=True, text=True).stdout.strip()))
for k, v in rows:
    print(f"{k:10s} {v}")
import gradio
assert int(gradio.__version__.split(".")[0]) >= 6, "Darkroom Studio needs Gradio 6 (theme/css are passed to launch())."

## 4 · Repository acquisition and licence inspection

Clones the Darkroom Studio branch of the ComfyUI-Darkroom fork. If a checkout already exists it is backed up (with a timestamp) before being updated, so local edits are never lost.

In [ ]:
import os, subprocess, time, shutil, sys
REPO_URL = "https://github.com/lawhutt27/ComfyUI-Darkroom.git"
BRANCH   = "claude/project-thread-b2tvuh"     # change to "main" once merged
ROOT     = "/content/ComfyUI-Darkroom" if IN_COLAB else os.path.abspath("ComfyUI-Darkroom")

if os.path.isdir(os.path.join(ROOT, ".git")):
    dirty = subprocess.run(["git", "-C", ROOT, "status", "--porcelain"], capture_output=True, text=True).stdout
    if dirty.strip():
        backup = f"{ROOT}_backup_{time.strftime('%Y%m%d_%H%M%S')}"
        shutil.copytree(ROOT, backup, symlinks=True, ignore=shutil.ignore_patterns("samples", "__pycache__"))
        print("Local changes found; backup saved to", backup)
    subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "merge", "-q", "--ff-only", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, ROOT], check=True)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

STUDIO = os.path.join(ROOT, "studio")
if STUDIO not in sys.path:
    sys.path.insert(0, STUDIO)

# ---- licences
import importlib.metadata as md, re
lic = [f for f in os.listdir(ROOT) if f.upper().startswith(("LICENSE", "LICENCE", "COPYING"))]
readme = open(os.path.join(ROOT, "README.md"), encoding="utf-8").read()
print("ComfyUI-Darkroom licence file:", lic or "NONE in repository",
      "| README states:", "MIT" if re.search(r"\bMIT\b", readme) else "not stated")
for dist in ["gradio", "rawpy", "numpy", "scipy", "pillow", "tifffile", "exifread"]:
    try:
        meta = md.metadata(dist)
        lic_text = meta.get("License-Expression") or ""
        if not lic_text:
            cls = [c.split("::")[-1].strip() for c in meta.get_all("Classifier") or [] if c.startswith("License ::")]
            short = (meta.get("License") or "").strip()
            lic_text = ", ".join(cls) or (short if short and len(short) < 40 and "\n" not in short else "see package")
        print(f"  {dist:10s} {lic_text}")
    except Exception:
        pass
print("  LibRaw     LGPL-2.1 or CDDL-1.0 (bundled inside the rawpy wheel)")
print("Full list: studio/docs/LICENSES.md")

## 5 · Image-processing engine

The engine (`darkroom_studio/engine.py`) works on float32 linear-light data. Each stage is skipped when its controls are neutral, so an untouched recipe is bit-exact. This cell runs a quick self-check.

In [ ]:
import numpy as np, time
from darkroom_studio import engine, recipe as R, colorspace as cms
from darkroom_studio import runtime
print(runtime.report())

rng = np.random.default_rng(0)
img = cms.srgb_decode(rng.random((400, 600, 3), dtype=np.float32))
ctx = engine.Context(cms.SRGB, (600, 400))
assert np.array_equal(engine.render(img, R.default_recipe(), ctx), img), "identity recipe must be bit-exact"
r = R.default_recipe(); r["basic"].update(exposure=0.5, contrast=20); r["hsl"]["green_saturation"] = -30
t = time.time(); out = engine.render(img, r, ctx)
print(f"engine OK: identity bit-exact; graded {out.shape} {out.dtype} in {(time.time()-t)*1000:.0f} ms")

## 6 · RAW decoder

RAW files are developed with LibRaw (via rawpy) to 16-bit linear data, separately from grading. White balance, exposure, highlight recovery, demosaic algorithm and working colour space are recorded in a sidecar. The RAW file itself is never modified.

In [ ]:
from darkroom_studio.raw import (RawSettings, HIGHLIGHT_OPTIONS, DEMOSAIC_OPTIONS, WHITE_BALANCE_OPTIONS,
                                 COLORSPACE_OPTIONS, BASE_CURVE_OPTIONS)
import rawpy
print("LibRaw", ".".join(map(str, rawpy.libraw_version)))
print("White balance :", WHITE_BALANCE_OPTIONS)
print("Highlights    :", HIGHLIGHT_OPTIONS)
print("Demosaic      :", DEMOSAIC_OPTIONS)
print("Colour spaces :", COLORSPACE_OPTIONS)
print("Base curve    :", BASE_CURVE_OPTIONS)
print("Defaults      :", RawSettings())

## 7 · Colour management

Sources are identified from their embedded ICC profile (or EXIF colour-space tag), linearised, and processed in their own primaries. Export either keeps the source profile (the original ICC bytes are embedded) or converts to sRGB, Display P3, Adobe RGB (1998) or ProPhoto RGB with a matching ICC profile. See `studio/docs/COLOUR_MANAGEMENT.md`.

In [ ]:
from darkroom_studio import colorspace as cms
for label, sp in cms.OUTPUT_SPACES.items():
    print(f"{label:18s} {sp.summary()}")
prof = cms.parse_icc(cms.ADOBE_RGB.icc_bytes())
assert prof.same_primaries(cms.ADOBE_RGB)
print("ICC write/parse round trip OK")

## 8 · Film emulation and LUT engine

In [ ]:
from darkroom_studio.film import FilmRegistry
from darkroom_studio.lut import LUTRegistry
films = FilmRegistry()
print(len(films.looks), "Studio looks:", ", ".join(n.replace("Studio / ", "") for n in films.looks))
print(len(films.darkroom), "additional stocks from the ComfyUI-Darkroom library (screened for tone reversals)")
print("These are aesthetic simulations, not measured reproductions of any film product.")

## 9 · Prompt interpreter

The default interpreter is local and offline. The optional OpenRouter mode sends **only the prompt text and the current recipe numbers**; the reply is validated against the schema and never executed.

To enable it, add a Colab secret named `OPENROUTER_API_KEY` (key icon in the left sidebar) and allow this notebook to read it. Never paste the key into a cell. Optionally set `OPENROUTER_MODEL`.

In [ ]:
from darkroom_studio.prompt import RuleInterpreter, get_api_key
from darkroom_studio import recipe as R
from darkroom_studio.session import Studio
Studio("/tmp/darkroom_selfcheck")            # fills the film/LUT registries used for validation
rules = RuleInterpreter()
for text in ["Make this photograph look like a warm 1970s analog film photograph with muted greens and soft highlights.",
             "Give this image a Kodak Portra-inspired appearance.",
             "Create a moody, atmospheric, desaturated appearance."]:
    r, notes, warns = rules.interpret(text, R.default_recipe())
    print("•", text, "\n   ", R.diff(r))
print("OpenRouter key available:", bool(get_api_key()), "(the key itself is never printed)")

## 10 · Gradio interface

Builds the interface without launching it (cell 13 launches). Layout: library and metadata on the left, the large preview with before/after in the centre, adjustment panels on the right, the prompt bar on top and a technical status bar at the bottom.

In [ ]:
import os
WORKDIR = "/content/darkroom_work" if IN_COLAB else os.path.abspath("darkroom_work")
# To keep originals, presets and exports between sessions, mount Drive first and use e.g.
# from google.colab import drive; drive.mount("/content/drive"); WORKDIR = "/content/drive/MyDrive/darkroom_work"
os.environ["DARKROOM_WORKDIR"] = WORKDIR
from darkroom_studio.ui.app import build_app
demo, studio = build_app(WORKDIR)
print("Interface built. Working folder:", WORKDIR)
print({k: v for k, v in studio.dirs.items()})
del demo

## 11 · Image export

Exports are rendered at full resolution from the float32 source in strips. Formats: 16-bit TIFF master (lossless, zlib), 16-bit and 8-bit PNG (lossless), JPEG quality 95–100 (4:4:4). The app warns when a format reduces bit depth or is lossy, and refuses to write a file whose dimensions differ from the source. This cell exports a synthetic image and reopens it.

In [ ]:
import numpy as np, tifffile, os
from darkroom_studio.session import Studio
st = Studio("/tmp/darkroom_selfcheck")
src = (np.random.default_rng(1).random((321, 457, 3)) * 65535).astype(np.uint16)
tifffile.imwrite("/tmp/darkroom_selfcheck/synthetic.tif", src, photometric="rgb")
s = st.open("/tmp/darkroom_selfcheck/synthetic.tif")
r = s.recipe; r["basic"]["contrast"] = 15; s.commit(r)
res = s.export("TIFF 16-bit (master, lossless)", "Same as source/working space")
back = tifffile.imread(res["path"])
print(res["path"], back.shape, back.dtype)
assert back.shape == (321, 457, 3) and back.dtype == np.uint16

## 12 · Automated tests

* **Synthetic tests** (always run): generated images covering import, export, colour management, editing, prompts, matching and presets.
* **Real-camera tests** (optional): genuine Nikon D5300 NEF and Canon EOS 1000D CR2 files (CC0, raw.pixls.us, about 90 MB, verified by SHA-256). They are skipped, not passed, if the files are absent.

In [ ]:
import subprocess, sys, os
RUN_CAMERA_TESTS = True       # downloads ~90 MB of sample RAW files once
RUN_SLOW_TESTS   = False      # full-resolution 24 MP exports, a few minutes on Colab CPU
if RUN_CAMERA_TESTS:
    subprocess.run([sys.executable, os.path.join(STUDIO, "tools", "fetch_samples.py"),
                    os.path.join(STUDIO, "samples")], check=False)
marker = [] if RUN_SLOW_TESTS else ["-m", "not slow"]
p = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-rs", *marker], cwd=STUDIO,
                   capture_output=True, text=True)
print(p.stdout[-6000:], p.stderr[-2000:])

## 13 · Launch

Opens the studio. In Colab a public `*.gradio.live` link is created (valid for 72 hours) because Colab cannot serve the app on localhost; anyone with the link can use the app while this runtime runs. Set `SHARE = False` to use Colab's inline output frame instead.

In [ ]:
SHARE = True
from darkroom_studio.ui.app import launch
launch(workdir=WORKDIR, share=SHARE, debug=False)